# 🚀 Huấn Luyện Toàn Bộ 6 Cấu Hình Ablation Study D2SA Trên NVIDIA A100 (40GB VRAM)

**Dự án:** Amodal Shape Prediction (Swin-UNet + SAM 2.1) — Mở rộng sang tập dữ liệu **MVTec D2SA**  
**Môi trường:** Google Colab Pro / Pro+ với GPU **NVIDIA A100 (40GB VRAM)**  
**Tính năng đặc biệt:**  
- 🔄 **Lưu trực tiếp vào Google Drive**: Toàn bộ Checkpoints (`.pth`), Text Logs (`.log`), Metrics (`.jsonl`), và Kết quả đánh giá (`.json`) được ghi thẳng vào Google Drive sau mỗi epoch với cơ chế `flush` + `fsync` nguyên tử.
- ⚡ **Tối ưu hóa A100 40GB**: Huấn luyện với `batch_size=16`, `accumulation_steps=1`, `num_workers=8`. VRAM tiêu thụ ~12-13 GB (chỉ chiếm ~30% VRAM của A100 40GB, loại bỏ 100% rủi ro OOM).
- 📊 **Tự động tổng hợp Table V**: Sau khi hoàn thành, tự động xuất bảng so sánh 6 cấu hình (mIoU, Dice, Precision, Recall, Invisible mIoU) chuẩn Markdown.

---

### 📌 Bước 1: Kiểm Tra GPU & Thông Số Phần Cứng
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 GPU -> High-RAM**.

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Name: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")

### 📁 Bước 2: Kết Nối Google Drive Để Lưu Trực Tiếp
Toàn bộ checkpoints và kết quả sẽ được lưu liên tục vào Google Drive để không bị mất khi Colab ngắt kết nối.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Thiết lập thư mục lưu trữ trên Google Drive của bạn
DRIVE_DIR = "/content/drive/MyDrive/amodal_shape_prediction"
os.makedirs(f"{DRIVE_DIR}/checkpoints/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs/d2sa", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results/d2sa", exist_ok=True)

print(f"✅ Đã kết nối Google Drive! Thư mục lưu trữ: {DRIVE_DIR}")

### 📥 Bước 3: Chuẩn Bị Mã Nguồn & Cài Đặt Thư Viện
Tự động nhận diện: Nếu bạn đã upload thư mục `amodal_shape_prediction` lên Google Drive thì copy sang SSD Colab để nạp code siêu nhanh, nếu chưa thì tự động clone từ GitHub.

In [ ]:
import os

DRIVE_REPO = "/content/drive/MyDrive/amodal_shape_prediction"
LOCAL_REPO = "/content/amodal_shape_prediction"
BRANCH = "feat/colab-a100"

# 1. Nếu đã upload thư mục lên Google Drive:
if os.path.exists(os.path.join(DRIVE_REPO, "scripts")):
    print("📁 Phát hiện repo có sẵn trên Google Drive! Sao chép sang SSD Colab...")
    os.makedirs(LOCAL_REPO, exist_ok=True)
    !cp -rn {DRIVE_REPO}/* {LOCAL_REPO}/ 2>/dev/null || cp -r {DRIVE_REPO}/* {LOCAL_REPO}/
    %cd {LOCAL_REPO}
# 2. Nếu repo cục bộ đã tồn tại (đã clone từ trước trên Colab):
elif os.path.exists(os.path.join(LOCAL_REPO, ".git")):
    print(f"🔄 Thư mục cục bộ đã tồn tại, đang fetch và checkout branch {BRANCH}...")
    %cd {LOCAL_REPO}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
# 3. Nếu chưa có trên Drive và chưa có cục bộ -> clone trực tiếp branch từ GitHub:
else:
    print(f"🌐 Chưa có repo, đang clone branch {BRANCH} từ GitHub...")
    !rm -rf {LOCAL_REPO}
    !git clone -b {BRANCH} https://github.com/vothenguyen/amodal_shape_prediction.git {LOCAL_REPO}
    %cd {LOCAL_REPO}

# Cài đặt các dependencies cần thiết (PyTorch đã có sẵn trong Colab)
!pip install -q timm albumentations pycocotools opencv-python tqdm

### 📦 Bước 4: Thiết Lập & Tự Động Kết Nối Dữ Liệu D2SA

> **Cơ chế nạp dữ liệu:** Dữ liệu cần nằm tại `/content/amodal_shape_prediction/data/D2SA/` gồm:
> - 3 file annotations: `D2S_amodal_training_rot0.json`, `D2S_amodal_augmented.json`, `D2S_amodal_validation.json`
> - Thư mục ảnh: `images/` (22,562 file ảnh .jpg)
>
> **Khuyên dùng:** Nếu bạn đã có sẵn thư mục `D2SA` trên Google Drive (ví dụ tại `MyDrive/amodal_shape_prediction/data/D2SA` hoặc `MyDrive/D2SA`), cell dưới đây sẽ **tự động phát hiện và kết nối qua symlink** trong tích tắc mà không cần tải lại!

In [ ]:
import os
import shutil

LOCAL_DATA_DIR = "/content/amodal_shape_prediction/data/D2SA"
os.makedirs("/content/amodal_shape_prediction/data", exist_ok=True)

# 1. Danh sách các vị trí ứng viên trên Google Drive
CANDIDATES = [
    "/content/drive/MyDrive/amodal_shape_prediction/data/D2SA",
    "/content/drive/MyDrive/D2SA",
    "/content/drive/MyDrive/data/D2SA",
    "/content/data/D2SA"
]

req_file = "D2S_amodal_training_rot0.json"
found_drive_path = None

for cand in CANDIDATES:
    if os.path.exists(os.path.join(cand, req_file)) and os.path.isdir(os.path.join(cand, "images")):
        found_drive_path = cand
        break

if found_drive_path:
    print(f"📁 Tìm thấy bộ dữ liệu D2SA đầy đủ trên Google Drive tại: {found_drive_path}")
    if os.path.exists(LOCAL_DATA_DIR) and os.path.islink(LOCAL_DATA_DIR):
        os.unlink(LOCAL_DATA_DIR)
    elif os.path.exists(LOCAL_DATA_DIR) and not os.listdir(LOCAL_DATA_DIR):
        os.rmdir(LOCAL_DATA_DIR)
    
    if not os.path.exists(LOCAL_DATA_DIR):
        print(f"🔗 Tạo symbolic link từ Google Drive sang SSD Colab: {LOCAL_DATA_DIR}...")
        os.symlink(found_drive_path, LOCAL_DATA_DIR, target_is_directory=True)
        print("✅ Kết nối dữ liệu thành công qua symlink!")
    else:
        print("✅ Thư mục cục bộ đã sẵn sàng dữ liệu.")
else:
    # Kiểm tra xem có file nén .zip / .tar.xz trên Drive không để giải nén
    zip_candidates = [
        "/content/drive/MyDrive/amodal_shape_prediction/data/D2SA.zip",
        "/content/drive/MyDrive/D2SA.zip",
        "/content/drive/MyDrive/d2s_amodal.zip",
    ]
    unzipped = False
    for zpath in zip_candidates:
        if os.path.exists(zpath):
            print(f"📦 Phát hiện file nén trên Drive: {zpath}. Đang giải nén vào {LOCAL_DATA_DIR}...")
            os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
            !unzip -q "{zpath}" -d "{LOCAL_DATA_DIR}"
            unzipped = True
            break
            
    if not unzipped:
        print("⚠️ Chưa phát hiện thư mục D2SA trên Google Drive!")
        print("💡 HƯỚNG DẪN ĐƯA DỮ LIỆU LÊN COLAB:")
        print("  Cách 1 (Khuyên dùng): Tải thư mục 'D2SA' (chứa images/ và 3 file JSON) lên Google Drive tại:")
        print("         /content/drive/MyDrive/amodal_shape_prediction/data/D2SA/")
        print("  Cách 2: Giải nén hoặc kéo thả trực tiếp thư mục D2SA vào ổ SSD Colab tại:")
        print("         /content/amodal_shape_prediction/data/D2SA/")


### 🔍 Bước 5: Kiểm Tra Tính Toàn Vẹn Dataset
Xác minh toàn bộ 3 file JSON annotations và 22,562 ảnh JPG.

In [ ]:
!python scripts/verify_d2sa_dataset.py

### 🧪 Bước 6 (Tùy chọn): Smoke Test Nhanh (1 Epoch trên 50 Mẫu)
Chạy thử nhanh 1 cấu hình để kiểm tra luồng nạp dữ liệu, forward pass, backward pass và tính metrics.

In [ ]:
!python scripts/run_ablation_d2sa.py \
    --rows 4 \
    --mode all \
    --epochs 1 \
    --subset-size 50 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 4 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

### 🏆 Bước 7: CHẠY 1 LẦN ĐẦY ĐỦ TOÀN BỘ 6 CẤU HÌNH (TABLE V) TRÊN A100 (40GB VRAM)

Lệnh dưới đây sẽ tự động huấn luyện tuần tự cả **6 cấu hình Ablation (Row 1 -> Row 6)**:
- **Epochs:** 30 epochs mỗi cấu hình
- **Batch size:** 16, **Accumulation steps:** 1 (Effective batch = 16, tốn ~12.5GB / 40GB VRAM)
- **Num workers:** 8 (Tối đa băng thông CPU)
- **Lưu trữ:** Mọi checkpoint, log và file metrics được đồng bộ tức thì vào Google Drive
- **Đánh giá:** Tự động đánh giá trên toàn bộ tập Validation (15,654 mẫu) và tổng hợp Table V

In [ ]:
!python scripts/run_ablation_d2sa.py \
    --rows 1 2 3 4 5 6 \
    --mode all \
    --epochs 30 \
    --batch-size 16 \
    --accumulation-steps 1 \
    --num-workers 8 \
    --device cuda \
    --drive-dir {DRIVE_DIR}

### 🔄 Bước 8: Tiếp Tục Huấn Luyện Khi Bị Gián Đoạn (Resume)
Nếu chẳng may Colab bị hết lượt hoặc ngắt kết nối giữa chừng, bạn chỉ cần nạp lại Drive và chạy lệnh này để tiếp tục chính xác từ epoch bị gián đoạn:

In [ ]:
# Ví dụ: Giả sử Row 4 bị dừng ở Epoch 12, tiếp tục chạy đến Epoch 30:
# !python scripts/run_ablation_d2sa.py \
#     --rows 4 \
#     --mode all \
#     --epochs 30 \
#     --batch-size 16 \
#     --accumulation-steps 1 \
#     --num-workers 8 \
#     --resume-epoch 12 \
#     --drive-dir {DRIVE_DIR}

### 📊 Bước 9: Xem Bảng Tổng Hợp Kết Quả Table V
Hiển thị bảng kết quả so sánh giữa 6 cấu hình sau khi hoàn tất.

In [ ]:
from IPython.display import display, Markdown
import os

summary_path = f"{DRIVE_DIR}/results/d2sa/table_v_summary.md"
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f"File bảng tổng hợp chưa tồn tại tại: {summary_path}")